In y axis, look at the median of 2 and 3 and comparing these within the cluster window (and other correlations)

As we have a lower sampling late, need to accommodate for misaligned timepoints. Cluster window buffered by 100ms to account for this, no rounding.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
import numpy as np
from scipy.stats import pearsonr, spearmanr


os.chdir('/Users/alex/Documents/action_hippo/action_hippo/eyes/deepmreye')
file = 'all_trials_flat.csv'
lower_lim = 1.191 - 0.2845 #make 1s window, same number of datapoints
upper_lim = 1.622 + 0.2845


In [3]:

# Load and clean data
df = pd.read_csv(file, index_col=None)
#df = df.drop('y', axis=1)
df['trial_type'] = pd.to_numeric(df['trial_type'], errors='coerce')
df['time_rel_acc'] = df['time_rel_acc'] - 0.5
df = df[(df['time_rel_acc'] <= upper_lim) & (df['time_rel_acc'] >= lower_lim)].copy() # window of 10 timepoints centred around cluster
# Step 1: Median x per trial
df_trial_medians = df.groupby(['subject', 'trial_id', 'trial_type'])[['y']].median().reset_index()
# Step 2: Count datapoints per trial
trial_counts = df.groupby(['subject', 'trial_id', 'trial_type']).size().reset_index(name='n_timepoints')
# Step 3: Compute average number of timepoints per trial per subject/trial_type
mean_counts = trial_counts.groupby(['subject', 'trial_type'])['n_timepoints'].median().reset_index()
num_timepoints = mean_counts['n_timepoints'].tolist()
# Step 4: Median x per subject/trial_type
df_subject_means = df_trial_medians.groupby(['subject', 'trial_type'])[['y']].median().reset_index()
# Step 5: Pivot to wide format
df_wide = df_subject_means.pivot(index='subject', columns='trial_type', values='y')
# Step 6: Keep only complete cases
df_wide = df_wide.dropna(subset=[2, 3]).rename(columns={2: 'y_type2', 3: 'y_type3'})
# Step 7: Paired one-sided t-test
tstat, pval_one_sided = ttest_rel(df_wide['y_type2'], df_wide['y_type3'], alternative='greater')

print(f"\n✅ One-sided paired t-test p-value (H1: median y_type2 > y_type3): {pval_one_sided:.5f}")
print(f"📊 Average number of timepoints per trial (per subject/trial_type): {num_timepoints}")


✅ One-sided paired t-test p-value (H1: median y_type2 > y_type3): 0.60549
📊 Average number of timepoints per trial (per subject/trial_type): [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0,